
# Demo: Create AI agent tools using Unity Catalog functions

[Create AI agent tools using Unity Catalog functions](https://docs.databricks.com/aws/en/generative-ai/agent-framework/create-custom-tool)

Use Unity Catalog functions to create AI agent tools that execute custom logic and perform specific tasks that extend the capabilities of LLMs beyond language generation.

Unity Catalog tools are really just [user-defined functions (UDFs) in Unity Catalog](https://docs.databricks.com/aws/en/udf/unity-catalog) under the hood:

* `create_python_function` accepts a Python callable.
* `create_function` accepts a SQL body create function statement (incl. [Python functions](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-sql-function#create-python-functions)).

## Requirements

🚨 NOTE: [Requirements](https://docs.databricks.com/aws/en/generative-ai/agent-framework/create-custom-tool#requirements) are not clear what the requirements are for creating and using UC functions. There are three paragraphs.

* Databricks Runtime 15.0 or later
* Python 3.10 or later
* [Serverless compute](https://docs.databricks.com/aws/en/compute/serverless/#requirements) must be enabled to execute Unity Catalog functions as AI agent tools in production.

## Unity Catalog functions vs. MCP servers

[When to use Unity Catalog functions vs. MCP servers](https://docs.databricks.com/aws/en/generative-ai/agent-framework/create-custom-tool#when-to-use-unity-catalog-functions-vs-mcp-servers)

In most cases, Databricks recommends MCP servers or defining the logic directly in agent code.

Databricks recommends using MCP servers to add Unity Catalog functions to your agent.

Use Unity Catalog functions as agent tools specifically for structured data retrieval tools when the query is known ahead of time and the agent provides the parameters.

## Create Unity Catalog Function

In [0]:
# Install Unity Catalog AI integration packages with the Databricks extras
# The Unity Catalog function integration is in the langchain-community package.
%pip install -U unitycatalog-ai[databricks]>=0.4.0 mlflow-skinny[databricks]>=3.12.0 databricks-agents>=1.10.2 databricks-mcp>=0.9.0
%restart_python

In [0]:
%pip show unitycatalog-ai

In [0]:
%pip show databricks-mcp

In [0]:
%pip show mlflow-skinny

In [0]:
%pip show databricks-agents

In [0]:
%pip show databricks-langchain

In [0]:
import mlflow

mlflow.autolog()


[Databricks Function Client](https://docs.unitycatalog.io/ai/client/#databricks-function-client)

`DatabricksFunctionClient` is a core component of the **Unity Catalog AI Core Library** to interact with Unity Catalog (UC) functions on Databricks.

In [0]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

client = DatabricksFunctionClient()

In [0]:
def add_numbers(number_1: float, number_2: float) -> float:
  """
  A function that accepts two floating point numbers adds them,
  and returns the resulting sum as a float.

  Args:
    number_1 (float): The first of the two numbers to add.
    number_2 (float): The second of the two numbers to add.

  Returns:
    float: The sum of the two input numbers.
  """
  return number_1 + number_2

In [0]:
%sql

CREATE CATALOG IF NOT EXISTS jacek_agents;
CREATE SCHEMA IF NOT EXISTS jacek_agents.default;

In [0]:
CATALOG = "jacek_agents"
SCHEMA = "default"

function_info = client.create_python_function(
  func=add_numbers,
  catalog=CATALOG,
  schema=SCHEMA,
  replace=True,
)
function_info

In [0]:
import ast
fun_props_dict = ast.literal_eval(function_info.properties)

import pandas as pd
props_df = pd.DataFrame(sorted(list(fun_props_dict.items())), columns=['key', 'value'])
display(props_df)

In [0]:
help(client.create_python_function)

In [0]:
result = client.execute_function(
  function_name=f"{CATALOG}.{SCHEMA}.add_numbers",
  parameters={"number_1": 36939.0, "number_2": 8922.4}
)

# client.execute_function returns a string (not a float)
assert float(result.value) == 45861.4


Once tested, add the Unity Catalog function to your agent.

## Add Unity Catalog function to Agent using MCP

[Add Unity Catalog functions to your agent](https://docs.databricks.com/aws/en/generative-ai/agent-framework/create-custom-tool#add-unity-catalog-functions-to-your-agent):

There are two approaches to add UC functions to agents:

* Using MCP (recommended)
* Using `UCFunctionToolkit` (to integrate with LangChain, LlamaIndex, OpenAI, Anthropic, and [other third party generative AI frameworks](https://docs.databricks.com/aws/en/generative-ai/agent-framework/unity-catalog-tool-integration))

The recommended MCP approach provides a simpler integration with automatic tool discovery and built-in authentication support.

Unity Catalog functions executed as AI agent tools require **serverless general compute** (Spark Connect serverless).

### Managed MCP URL for Unity Catalog functions

The managed MCP URL for Unity Catalog functions is: `https://<workspace-hostname>/api/2.0/mcp/functions/{catalog}/{schema}`.

The endpoint above exposes the schema, meaning that all functions created within that schema will be exposed as individual tools of that managed MCP server.

You can optionally specify a specific function by appending `/{function_name}`.

Connect your agent to Unity Catalog functions through MCP.


## Databricks MCP endpoint

The Databricks MCP endpoint follows the [MCP protocol](https://modelcontextprotocol.io/docs/getting-started/intro) (which uses JSON-RPC 2.0 over Streamable HTTP).

The key methods are:

* `tools/list` — discovers all functions in the schema (no params needed)
* `tools/call` — executes a function, with `params.name` as the function name and `params.arguments` as the input map

The `Accept` header should include `application/json` or `text/event-stream` (for streaming responses).

Authentication is a standard Databricks PAT or OAuth token.


### List available tools

<br>

```shell
databricks api post /api/2.0/mcp/functions/jacek_agents/default \
    --debug \
    --profile u01-ingestion \
    --json '{"jsonrpc": "2.0", "id": 1, "method": "tools/list"}'
```


### Call a tool (invoke add_numbers)

<br>

```shell
databricks api post /api/2.0/mcp/functions/jacek_agents/default/add_numbers \
    --debug \
    --profile u01-ingestion \
    --json '{"jsonrpc": "2.0", "id": 1,
    "method": "tools/call",
    "params": {
    "name": "jacek_agents__default__add_numbers",
    "arguments": {"number_1": 7.2, "number_2": 7.8}}}'
```

In [0]:
import requests
import json
from databricks.sdk import WorkspaceClient

ws = WorkspaceClient()
host = ws.config.host

# Get the notebook context token (works on serverless)
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
token = ctx.apiToken().get()

CATALOG = "jacek_agents"
SCHEMA = "default"
url = f"{host}/api/2.0/mcp/functions/{CATALOG}/{SCHEMA}"

response = requests.post(
    url,
    headers={
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
        "Accept": "application/json",
    },
    json={"jsonrpc": "2.0", "id": 1, "method": "tools/list"},
)

print(json.dumps(response.json(), indent=2))


Key observations from the response:

* Tool name uses double-underscore separators: `jacek_agents__default__add_numbers`
* `inputSchema` describes the two `number` parameters with descriptions pulled from the Python docstring
* `outputSchema` returns a tabular format (`columns` + `rows` + `is_truncated` flag)
* The `annotations` field includes catalog, schema, language, and hint flags


## Step 1: Create a single-turn LLM call

[Step 1: Create a single-turn LLM call](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/supervisor-api#step-1-create-a-single-turn-llm-call)

### Deploy Agent on Model Serving for GenAI Apps

[Databricks recommends deploying agents on Databricks Apps for full control over agent code, server configuration, and deployment workflow](https://docs.databricks.com/aws/en/generative-ai/agent-framework/deploy-agent)

Before deploying an agent, you have to register the agent to Unity Catalog.

Registering the agent packages it as a model in Unity Catalog.
As a result, you can use Unity Catalog permissions for authorization for resources in the agent.

## Databricks MCP Python API

[Databricks MCP Python API](https://api-docs.databricks.com/python/databricks-ai-bridge/latest/databricks_mcp.html)

In [0]:
# Databricks notebooks already run inside an active asyncio event loop
# DatabricksMCPClient.list_tools() internally calls asyncio.run()
# and fails with RuntimeError: asyncio.run() cannot be called from a running event loop
import nest_asyncio
nest_asyncio.apply()

### DatabricksMCPClient

[DatabricksMCPClient](https://api-docs.databricks.com/python/databricks-ai-bridge/latest/databricks_mcp.html#databricks_mcp.DatabricksMCPClient) is a client for interacting with a MCP (Model Context Protocol) servers on Databricks.

This class provides a simplified interface to communicate with a specified MCP server URL with Databricks Authorization.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient

workspace_client = WorkspaceClient()
host = workspace_client.config.host

# Connect to the UC functions MCP server
mcp_client = DatabricksMCPClient(
    # The base URL of the MCP server to which this client connects.
    server_url=f"{host}/api/2.0/mcp/functions/{CATALOG}/{SCHEMA}",
    # The Databricks workspace client used for authentication and requests.
    workspace_client=workspace_client,
)

In [0]:
mcp_tools = mcp_client.list_tools()
mcp_tools

In [0]:
call_tool_result = mcp_client.call_tool(
    tool_name="jacek_agents__default__add_numbers",
    arguments={
        "number_1": 2.3,
        "number_2": 2.7,
    },
)
call_tool_result

In [0]:
# If authoring a custom code agent that runs tools from a Databricks Managed MCP server,
# call this method and pass the returned resources to mlflow.pyfunc.log_model when logging your agent,
# to enable your agent to authenticate to the MCP server and run tools when deployed.

databricks_resources = mcp_client.get_databricks_resources()
databricks_resources


## MLflow's Model Resources

[ResourceType](https://www.mlflow.org/docs/latest/api_reference/_modules/mlflow/models/resources.html) defines the different types of resources needed to serve a model.

There's `DatabricksResource` to defines all the Databricks resources to serve a model.

Example usage: [Log and register AI agents (Model Serving)](https://docs.databricks.com/en/generative-ai/log-agent.html#specify-resources-for-pyfunc-or-langchain-agent)

In [0]:
db_func = databricks_resources[0]
assert db_func.target_uri == 'databricks'

db_func.to_dict()

In [0]:
from mlflow.models.resources import DatabricksFunction

db_func.type

In [0]:
dbutils.notebook.exit('Stop here')

## Databricks Langchain Integrations Python API

[Databricks Langchain Integrations Python API](https://api-docs.databricks.com/python/databricks-ai-bridge/latest/databricks_langchain.html)

Unity Catalog OSS in [Using Unity Catalog AI with LangChain](https://docs.unitycatalog.io/ai/integrations/langchain/#using-unity-catalog-ai-with-langchain)

In [0]:
%skip
%pip install -U databricks-langchain>=0.19.0 langchain-community>=0.4.1 langchain>=1.3.1 langgraph>=1.2.0

### DatabricksMultiServerMCPClient

In [0]:
from databricks_langchain import (
    DatabricksMultiServerMCPClient,
    DatabricksMCPServer
)

databricks_mcp_client = DatabricksMultiServerMCPClient(
    # servers: List of MCPServer or DatabricksMCPServer configurations
    servers = [
        DatabricksMCPServer(
            name="add_numbers",
            url=f"{host}/api/2.0/mcp/functions/{CATALOG}/{SCHEMA}",
            workspace_client=workspace_client,
        ),
    ]
)

In [0]:
help(DatabricksMultiServerMCPClient)

In [0]:
tools = await databricks_mcp_client.get_tools()
print(f"List of LangChain tools: {tools}")
# databricks_mcp_client.execute(
#     function_name="add_numbers",
#     parameters={"number_1": 36939.0, "number_2": 8922.4}
# )


## Agents

LangChain's [Agents](https://docs.langchain.com/oss/python/langchain/agents):

Agents combine language models with tools to create systems that can reason about tasks, decide which tools to use, and iteratively work towards solutions.

In [0]:
from databricks_langchain import (
    ChatDatabricks,
    DatabricksMCPServer,
    DatabricksMultiServerMCPClient,
)


LLM_ENDPOINT_NAME = "databricks-claude-sonnet-4-5"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)

In [0]:
%pip show langchain

In [0]:
%pip show langgraph

In [0]:
from langchain_core.messages import HumanMessage, ToolMessage

class SimpleToolCallingAgent:
    def __init__(self, model, tools, system_prompt="You are a helpful assistant"):
        self.model = model.bind_tools(tools)
        self.tools = {tool.name: tool for tool in tools}
        self.system_prompt = system_prompt

    def invoke(self, user_input):
        messages = [
            {"role": "system", "content": self.system_prompt},
            HumanMessage(content=user_input),
        ]
        while True:
            response = self.model.invoke(messages)
            messages.append(response)
            if not response.tool_calls:
                return response.content
            for tool_call in response.tool_calls:
                tool = self.tools[tool_call["name"]]
                result = tool.invoke(tool_call["args"])
                messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))

In [0]:
import mlflow
mlflow.langchain.autolog()

In [0]:
AGENT = SimpleToolCallingAgent(llm, tools, "You are a helpful assistant")

In [0]:
import mlflow

# Model should either be an instance of PyFuncModel, Langchain type, or LlamaIndex index.
# mlflow.models.set_model(AGENT)

In [0]:
AGENT.invoke("What is 7.2 + 7.8?")


## Supervisor API
If your agent uses only Databricks-hosted tools and does not need custom logic between tool calls, you can use the [Supervisor API (Beta)](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/supervisor-api) to let Databricks manage the agent loop for you.

The Supervisor API simplifies building custom agents on Databricks.